# 11. Compare bulkhic

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
import glob
import cooler
import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
from collections import defaultdict

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")



In [ ]:
indir = f'{ENTEX_ROOT}/'
loop_dir = f'{indir}loop/subtype/'
hic_dir = f'{indir}bulk_HiC/data/'


In [ ]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
from scipy.stats import zscore

def dist_norm(A, cap=5, max_dist=5000000, res=10000):
    n_diag = max_dist // res + 1
    E = np.zeros(A.shape, dtype=float)
    mask = np.zeros(E.shape, dtype=bool)
    row, col = np.diag_indices(E.shape[0])
    mask[row, col] = True
    for i in range(1, n_diag):
        mask[row[:-i], col[i:]] = True

    E[row, col] = 0
    for i in range(1, n_diag):
        tmp = A.diagonal(i).copy()
        tmp_filter = (tmp > 0)
        tmp2 = tmp[tmp_filter]
        if len(tmp2) == 0:
            E[row[:-i], col[i:]] = 0
        else:
            cutoff = np.percentile(tmp2, 99)
            tmp2 = np.where(tmp2 < cutoff, tmp2, cutoff)
            tmp2 = zscore(tmp2)
            tmp2[np.isnan(tmp2)] = 0
            tmp2 = np.clip(tmp2, a_min=-cap, a_max=cap)
            tmp[~tmp_filter] = tmp2.min()
            tmp[tmp_filter] = tmp2.copy()
            E[row[:-i], col[i:]] = tmp.copy()
    return E


In [ ]:
hic_list = pd.read_csv(f'{hic_dir}metadata.tsv', sep='\t', header=0, index_col=0)
hic_list

In [ ]:
res = 10000
loop_all = pd.read_csv(f'{loop_dir}merged_loop_subtype.bedpe', header=None, index_col=None, sep='\t')


In [ ]:
loop_A = pd.DataFrame(index=loop_all.index, columns=hic_list.index[hic_list['File format']=='hic'])
loop_E = loop_A.copy()

loop_all = {chrom:loop_all.loc[loop_all[0]==chrom, [1,4]] // res for chrom in chrom_sizes.index[:22]}


In [ ]:
from scipy.sparse import csr_matrix
import hicstraw

def load_hic(hic_path, chrom, res=10000):
    data = hicstraw.straw("observed", "VC_SQRT", hic_path, chrom, chrom, "BP", res)
    nbins = (chrom_sizes.loc[chrom] // res) + 1
    idx, idy, counts = [], [], []
    for xx in data:
        idx.append(xx.binX // res)
        idy.append(xx.binY // res)
        counts.append(xx.counts)
    A = csr_matrix((counts, (idx, idy)), shape=(nbins, nbins)).toarray()
    return A


In [ ]:
def load_loop_values(hic_path, chrom, res=10000):
    A = load_hic(hic_path, chrom)
    E = dist_norm(A)
    idx, idy = loop_all[chrom].values.T
    return [A[idx, idy], E[idx, idy]]


In [ ]:
# hic_file = 'ENCFF736ITL'
# hic_path = f'{hic_dir}{hic_file}.hic'
# chrom = 'chr2'
# tmp = load_loop_values(hic_path, chrom)


In [ ]:
# loop_A.loc[loop_all[chrom].index, hic_file] = tmp[0].copy()
# loop_E.loc[loop_all[chrom].index, hic_file] = tmp[1].copy()


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

cpu = 24
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for hic_file in hic_list.loc[hic_list['File format'] == 'hic'].index:
        for chrom in chrom_sizes.index[:22]:
            future = executor.submit(
                load_loop_values,
                hic_path=f'{hic_dir}{hic_file}.hic',
                chrom=chrom,
            )
            futures[future] = [chrom, hic_file]

    for future in as_completed(futures):
        tmp = future.result()
        chrom, hic_file = futures[future] 
        loop_A.loc[loop_all[chrom].index, hic_file] = tmp[0].copy()
        loop_E.loc[loop_all[chrom].index, hic_file] = tmp[1].copy()
        print(f'{chrom} {hic_file} finished')
        

In [ ]:
print(loop_A.shape, loop_E.shape)

In [ ]:
loop_A.to_feather(f'{hic_dir}../merged_loop_raw.feather')
loop_E.to_feather(f'{hic_dir}../merged_loop_norm.feather')


In [ ]:
loop_A = pd.read_feather(f'{hic_dir}../merged_loop_raw.feather')
loop_E = pd.read_feather(f'{hic_dir}../merged_loop_norm.feather')


In [ ]:
loop_meta = pd.read_csv(f'{indir}subtype_meta.tsv', sep='\t', header=0, index_col=0)
loop_all = pd.read_csv(f'{loop_dir}merged_loop_subtype.bedpe', header=None, index_col=None, sep='\t')
loop_all.index = loop_all[0].astype(str) + '-' + (loop_all[1] // res).astype(str) + '-' + (loop_all[4] // res).astype(str)


In [ ]:
# def load_loop(ct):
#     global loop_subtype
#     tmp = pd.read_csv(f'{loop_dir}{ct}.loop.bedpe', header=None, index_col=None, sep='\t')
#     idx = tmp[0].astype(str) + '-' + (tmp[1] // res).astype(str) + '-' + (tmp[4] // res).astype(str)
#     return idx, tmp[6].values


In [ ]:
# loop_subtype = pd.DataFrame(0, index=loop_all.index, columns=loop_meta.index)

# cpu = 24
# with ProcessPoolExecutor(cpu) as executor:
#     futures = {}
#     for ct in loop_meta.index:
#         future = executor.submit(
#             load_loop,
#             ct=ct
#         )
#         futures[future] = ct

#     for future in as_completed(futures):
#         idx, tmp = future.result()
#         ct = futures[future]
#         loop_subtype.loc[idx, ct] = tmp.copy()
#         print(f'{ct} finished')
        

In [ ]:
# loop_subtype = loop_subtype.reset_index(drop=True)
# loop_subtype.to_feather(f'{hic_dir}../merged_loop_subtype.feather')


In [ ]:
loop_fdr = pd.read_hdf(f'{hic_dir}../merged_loop.fdr.hdf')
loop_fdr.index = loop_fdr[0].astype(str) + '-' + (loop_fdr[1] // res).astype(str) + '-' + (loop_fdr[4] // res).astype(str)
loop_fdr


In [ ]:
# loop_subtype = pd.read_feather(f'{hic_dir}../merged_loop_subtype.feather')
# ave = loop_subtype.mean(axis=1)
# stdev = loop_subtype.std(axis=1)

In [ ]:
loop_A = pd.read_feather(f'{hic_dir}../merged_loop_raw.feather')
loop_N = pd.read_feather(f'{hic_dir}../merged_loop_norm.feather')


In [ ]:
loop_Q = pd.read_hdf(f'{hic_dir}../loop_Q.hdf')
loop_E = pd.read_hdf(f'{hic_dir}../loop_E.hdf')


In [ ]:
from scipy.stats import norm, zscore

thres1 = norm.isf(0.025)
thres2 = norm.isf(0.15)
print(thres1, thres2)


In [ ]:
selp = ((zscore(loop_fdr['Qanova'])>thres2) & (zscore(loop_fdr['Tanova'])>thres2))
print(selp.sum())

In [ ]:
loop_A.index = loop_all.index.copy()
loop_N.index = loop_all.index.copy()
loop_Q.index = loop_fdr.index.copy()
loop_E.index = loop_fdr.index.copy()


In [ ]:
selp = selp.index[selp]


In [ ]:
# result = pd.DataFrame(index=loop_E.columns)
# for ct in loop_meta.index:
#     selp = (loop_subtype[ct] - ave).nlargest(5000).index
#     result[ct] = loop_E.loc[selp].mean(axis=0)



In [ ]:
# from sklearn.metrics import pairwise_distances
# selp = stdev.nlargest(100000).index
# dist = pairwise_distances(loop_E.loc[selp].T, loop_subtype.loc[selp].T, metric='correlation')


In [ ]:
from sklearn.metrics import pairwise_distances
# selp = stdev.nlargest(100000).index
# dist = pairwise_distances(loop_A.loc[selp].T, loop_Q.loc[selp].T, metric='correlation')
dist = pairwise_distances(loop_N.loc[selp].T, loop_E.loc[selp].T, metric='correlation')
dist = pd.DataFrame(dist, 
                    index=loop_A.columns.map(hic_list['Biosample term name']), 
                    columns=loop_Q.columns.map(loop_meta['celltype_L2_both_abbr']))
dist.to_hdf(f'compare_bulkhic/Ecorr_subtype_ENCODE.hdf', key='data')


In [ ]:
dist = pd.read_hdf(f'compare_bulkhic/Ecorr_subtype_ENCODE.hdf', key='data')
cg = sns.clustermap(1-dist, z_score=0, xticklabels=1, yticklabels=1, figsize=(36, 18), cmap='cividis_r', vmin=1, vmax=3)
rorder = cg.dendrogram_row.reordered_ind
corder = cg.dendrogram_col.reordered_ind


In [ ]:
# dist = dist.iloc[rorder[::-1], :]
# bestbulk = np.argmin(dist, axis=0)
# dist = dist.iloc[:, np.argsort(bestbulk)]

dist = dist.iloc[:, corder[::-1]]
bestbulk = np.argmin(dist, axis=1)
dist = dist.iloc[np.argsort(bestbulk), :]


In [ ]:
fig, ax = plt.subplots(figsize=(12,12), dpi=300)
ax.imshow(zscore(1-dist, axis=1), vmin=1, vmax=3, cmap='cividis_r')
xticks = np.arange(0, dist.shape[1], 2)
ax.set_xticks(xticks)
ax.set_xticklabels(dist.columns[xticks], fontsize=8, rotation=90)
yticks = np.arange(0, dist.shape[0], 2)
ax.set_yticks(yticks)
ax.set_yticklabels(dist.index[yticks], fontsize=8, rotation=0)
fig.savefig('compare_bulkhic/Qcorr_subtype_ENCODE.pdf', transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(12,12), dpi=300)
ax.imshow(zscore(1-dist, axis=1), vmin=1, vmax=3, cmap='cividis_r')
xticks = np.arange(0, dist.shape[1], 2)
ax.set_xticks(xticks)
ax.set_xticklabels(dist.columns[xticks], fontsize=8, rotation=90)
yticks = np.arange(0, dist.shape[0], 2)
ax.set_yticks(yticks)
ax.set_yticklabels(dist.index[yticks], fontsize=8, rotation=0)
fig.savefig('compare_bulkhic/Ecorr_subtype_ENCODE.pdf', transparent=True)
